# 03 - Collect and compare results

Aggregates all freeform eval and teacher-forced results produced by
`02_eval_all.ipynb` into combined pandas DataFrames, then produces:

1. A **3×3 accuracy heatmap** (line plots) — rows = calibration benchmark,
   cols = eval benchmark, x = sparsity level, y = pass@1 / accuracy.
2. A **3×3 freeform perplexity heatmap** — same layout, y = average
   teacher-forced perplexity of the gold answer (measured during freeform eval).
3. A **3×3 TF perplexity heatmap** — mean perplexity across all test records
   from the standalone teacher-forced logit collection.
4. A **cross-dataset comparison** — for each eval benchmark, overlay lines
   from each calibration source to visualise transfer effects.

DataFrames are saved to `notebooks/experiment/results/` as CSV files.


In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env", override=False)
except ImportError:
    pass

print("REPO_ROOT =", REPO_ROOT)


## Load experiment config

Reads `experiment_config.json` for S3 URIs. If the file is missing, set
`FREEFORM_EVAL_URIS` and `TEACHER_FORCED_URIS` manually.


In [ ]:
import json
import boto3

AWS_PROFILE = os.environ.get("AWS_PROFILE", "rengz")
RESULTS_BUCKET = os.environ.get(
    "RESULTS_BUCKET", "pruning-metrics-results-414266451290"
)

NOTEBOOK_DIR = Path(__file__).parent
experiment_config_path = NOTEBOOK_DIR / "experiment_config.json"

if experiment_config_path.exists():
    _cfg = json.loads(experiment_config_path.read_text(encoding="utf-8"))
    FREEFORM_EVAL_URIS  = _cfg.get("freeform_eval_uris", {})
    TEACHER_FORCED_URIS = _cfg.get("teacher_forced_uris", {})
else:
    FREEFORM_EVAL_URIS  = {}
    TEACHER_FORCED_URIS = {}

assert FREEFORM_EVAL_URIS, (
    "No freeform eval URIs found. Run 02_eval_all.ipynb first, or set "
    "FREEFORM_EVAL_URIS manually."
)
assert TEACHER_FORCED_URIS, (
    "No teacher-forced URIs found. Run 02_eval_all.ipynb first, or set "
    "TEACHER_FORCED_URIS manually."
)

# Expected key format: "<cal_label>_<eval_label>" e.g. "gsm8k_humaneval"
print(f"Freeform eval runs:    {len(FREEFORM_EVAL_URIS)}")
print(f"Teacher-forced runs:   {len(TEACHER_FORCED_URIS)}")
for k in sorted(FREEFORM_EVAL_URIS):
    print(f"  {k}: {FREEFORM_EVAL_URIS[k]}")


## Aggregate freeform eval summaries

Downloads `summary.json` from each of the 9 freeform eval runs and merges
them into a single DataFrame with columns for calibration dataset, eval
dataset, pruning level, accuracy, and perplexity.


In [ ]:
import pandas as pd

session = boto3.session.Session(profile_name=AWS_PROFILE)
s3 = session.client("s3")

BENCHMARK_LABELS = {
    "gsm8k":         "GSM8K",
    "humaneval":     "HumanEval+",
    "arc_challenge": "ARC-Challenge",
}

def _s3_get_json(uri: str) -> dict:
    """Download a single S3 JSON object given a full s3:// URI."""
    assert uri.startswith("s3://")
    rest = uri[5:]
    bucket, key = rest.split("/", 1)
    key = key.rstrip("/") + "/summary.json"
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return json.loads(body.decode("utf-8"))

freeform_rows = []
for combo_key, uri in FREEFORM_EVAL_URIS.items():
    cal_label, eval_label = combo_key.split("_", 1)
    try:
        summary = _s3_get_json(uri)
    except Exception as exc:
        print(f"  SKIP {combo_key}: {exc}")
        continue
    for entry in summary.get("levels", []):
        freeform_rows.append({
            "cal_dataset":        cal_label,
            "eval_dataset":       eval_label,
            "cal_dataset_label":  BENCHMARK_LABELS.get(cal_label, cal_label),
            "eval_dataset_label": BENCHMARK_LABELS.get(eval_label, eval_label),
            "pruning_level":      entry["pruning_level"],
            "num_test_tasks":     entry["num_test_tasks"],
            "num_passed":         entry.get("num_passed"),
            "pass_at_1":          entry.get("pass_at_1"),
            "average_perplexity": entry.get("average_perplexity"),
            "elapsed_seconds":    entry.get("elapsed_seconds"),
        })

freeform_df = (
    pd.DataFrame(freeform_rows)
    .sort_values(["cal_dataset", "eval_dataset", "pruning_level"])
    .reset_index(drop=True)
)
print(f"Freeform eval rows: {len(freeform_df)}")
freeform_df.head(15)


## Accuracy vs. pruning level (3×3 grid)

Each subplot shows how accuracy (pass@1 for coding, exact-match for math/MCQ)
changes with sparsity for one (calibration, eval) pair.


In [ ]:
import matplotlib.pyplot as plt

cal_labels  = sorted(freeform_df["cal_dataset"].unique())
eval_labels = sorted(freeform_df["eval_dataset"].unique())
n_cal  = len(cal_labels)
n_eval = len(eval_labels)

fig, axes = plt.subplots(n_cal, n_eval, figsize=(5 * n_eval, 4 * n_cal),
                          sharex=True)

for row_i, cal in enumerate(cal_labels):
    for col_j, ev in enumerate(eval_labels):
        ax = axes[row_i][col_j]
        subset = freeform_df[
            (freeform_df["cal_dataset"] == cal) &
            (freeform_df["eval_dataset"] == ev)
        ].sort_values("pruning_level")

        if subset.empty or subset["pass_at_1"].isna().all():
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes)
        else:
            ax.plot(subset["pruning_level"], subset["pass_at_1"],
                    "o-", linewidth=2)
            ymax = max(0.05, float(subset["pass_at_1"].max()) * 1.1)
            ax.set_ylim(0.0, ymax)

        if row_i == 0:
            ax.set_title(BENCHMARK_LABELS.get(ev, ev), fontsize=11, fontweight="bold")
        if col_j == 0:
            ax.set_ylabel(
                f"cal: {BENCHMARK_LABELS.get(cal, cal)}\npass@1 / accuracy",
                fontsize=9,
            )
        if row_i == n_cal - 1:
            ax.set_xlabel("Pruning level (% sparsity)", fontsize=9)
        ax.grid(True, alpha=0.3)

fig.suptitle("Accuracy vs. sparsity — calibration (rows) × eval (cols)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig


## Freeform perplexity vs. pruning level (3×3 grid)

Perplexity of the ground-truth answer under teacher forcing, measured during
the freeform eval run (one extra forward pass per test record per level).
Lower = model assigns higher probability to the correct answer.


In [ ]:
fig, axes = plt.subplots(n_cal, n_eval, figsize=(5 * n_eval, 4 * n_cal),
                          sharex=True)

for row_i, cal in enumerate(cal_labels):
    for col_j, ev in enumerate(eval_labels):
        ax = axes[row_i][col_j]
        subset = freeform_df[
            (freeform_df["cal_dataset"] == cal) &
            (freeform_df["eval_dataset"] == ev)
        ].sort_values("pruning_level")
        ppl = subset["average_perplexity"].dropna()

        if ppl.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes)
        else:
            ax.plot(subset.loc[ppl.index, "pruning_level"], ppl,
                    "s--", color="tab:orange", linewidth=2)

        if row_i == 0:
            ax.set_title(BENCHMARK_LABELS.get(ev, ev), fontsize=11, fontweight="bold")
        if col_j == 0:
            ax.set_ylabel(
                f"cal: {BENCHMARK_LABELS.get(cal, cal)}\nperplexity",
                fontsize=9,
            )
        if row_i == n_cal - 1:
            ax.set_xlabel("Pruning level (% sparsity)", fontsize=9)
        ax.grid(True, alpha=0.3)

fig.suptitle("Freeform-eval perplexity (gold answer) — calibration (rows) × eval (cols)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig


## Aggregate teacher-forced summaries

Downloads `summary.json` from each of the 9 TF runs and computes per-level
mean perplexity and mean average log-probability across all scored test
records.


In [ ]:
def _s3_get_tf_json(uri: str) -> dict:
    """Download teacher-forced summary.json given a base s3:// URI."""
    assert uri.startswith("s3://")
    rest = uri[5:]
    bucket, key = rest.split("/", 1)
    key = key.rstrip("/") + "/summary.json"
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    return json.loads(body.decode("utf-8"))

tf_rows = []
for combo_key, uri in TEACHER_FORCED_URIS.items():
    cal_label, eval_label = combo_key.split("_", 1)
    try:
        summary = _s3_get_tf_json(uri)
    except Exception as exc:
        print(f"  SKIP {combo_key}: {exc}")
        continue

    samples = summary.get("samples", {})
    # Build a per-(level, sample) table then aggregate to per-level means.
    level_buckets: dict[float, list] = {}
    for task_id, payload in samples.items():
        for entry in payload.get("by_level", []):
            lv = entry["pruning_level"]
            level_buckets.setdefault(lv, []).append({
                "perplexity":    entry.get("perplexity"),
                "average_logprob": entry.get("average_logprob"),
                "num_tokens":    entry.get("num_answer_tokens"),
            })

    for lv, entries in sorted(level_buckets.items()):
        ppls  = [e["perplexity"]    for e in entries if e["perplexity"] is not None]
        logps = [e["average_logprob"] for e in entries if e["average_logprob"] is not None]
        tf_rows.append({
            "cal_dataset":          cal_label,
            "eval_dataset":         eval_label,
            "cal_dataset_label":    BENCHMARK_LABELS.get(cal_label, cal_label),
            "eval_dataset_label":   BENCHMARK_LABELS.get(eval_label, eval_label),
            "pruning_level":        lv,
            "num_samples":          len(entries),
            "mean_perplexity":      sum(ppls) / len(ppls) if ppls else None,
            "mean_avg_logprob":     sum(logps) / len(logps) if logps else None,
        })

tf_df = (
    pd.DataFrame(tf_rows)
    .sort_values(["cal_dataset", "eval_dataset", "pruning_level"])
    .reset_index(drop=True)
)
print(f"TF rows: {len(tf_df)}")
tf_df.head(15)


## Teacher-forced perplexity vs. pruning level (3×3 grid)

Mean perplexity across all test records per (calibration, eval, level)
combination from the standalone TF runs.


In [ ]:
fig, axes = plt.subplots(n_cal, n_eval, figsize=(5 * n_eval, 4 * n_cal),
                          sharex=True)

for row_i, cal in enumerate(cal_labels):
    for col_j, ev in enumerate(eval_labels):
        ax = axes[row_i][col_j]
        subset = tf_df[
            (tf_df["cal_dataset"] == cal) &
            (tf_df["eval_dataset"] == ev)
        ].sort_values("pruning_level")
        ppl = subset["mean_perplexity"].dropna()

        if ppl.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes)
        else:
            ax.plot(subset.loc[ppl.index, "pruning_level"], ppl,
                    "^-", color="tab:green", linewidth=2)

        if row_i == 0:
            ax.set_title(BENCHMARK_LABELS.get(ev, ev), fontsize=11, fontweight="bold")
        if col_j == 0:
            ax.set_ylabel(
                f"cal: {BENCHMARK_LABELS.get(cal, cal)}\nmean perplexity",
                fontsize=9,
            )
        if row_i == n_cal - 1:
            ax.set_xlabel("Pruning level (% sparsity)", fontsize=9)
        ax.grid(True, alpha=0.3)

fig.suptitle("Teacher-forced mean perplexity — calibration (rows) × eval (cols)",
             fontsize=13, fontweight="bold", y=1.01)
fig.tight_layout()
fig


## Cross-dataset transfer analysis

For each eval benchmark, overlay accuracy lines from all three calibration
sources to show whether domain-specific pruning transfers across tasks.


In [ ]:
COLORS = {
    "gsm8k":         "tab:blue",
    "humaneval":     "tab:orange",
    "arc_challenge": "tab:green",
}
MARKERS = {"gsm8k": "o", "humaneval": "s", "arc_challenge": "^"}

fig, axes = plt.subplots(1, n_eval, figsize=(5 * n_eval, 4), sharey=False)

for col_j, ev in enumerate(eval_labels):
    ax = axes[col_j]
    for cal in cal_labels:
        subset = freeform_df[
            (freeform_df["cal_dataset"] == cal) &
            (freeform_df["eval_dataset"] == ev)
        ].sort_values("pruning_level")
        if subset.empty or subset["pass_at_1"].isna().all():
            continue
        ax.plot(
            subset["pruning_level"], subset["pass_at_1"],
            marker=MARKERS.get(cal, "x"),
            color=COLORS.get(cal, "gray"),
            linewidth=2,
            label=f"pruned on {BENCHMARK_LABELS.get(cal, cal)}",
        )
    ax.set_title(f"Eval: {BENCHMARK_LABELS.get(ev, ev)}", fontsize=11,
                 fontweight="bold")
    ax.set_xlabel("Pruning level (% sparsity)")
    ax.set_ylabel("pass@1 / accuracy")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Cross-dataset transfer: accuracy by calibration source",
             fontsize=13, fontweight="bold", y=1.02)
fig.tight_layout()
fig


## Save results to CSV

Writes both DataFrames to `notebooks/experiment/results/` for offline
analysis or sharing.


In [ ]:
results_dir = NOTEBOOK_DIR / "results"
results_dir.mkdir(exist_ok=True)

freeform_csv = results_dir / "freeform_eval.csv"
tf_csv       = results_dir / "teacher_forced.csv"

freeform_df.to_csv(freeform_csv, index=False)
tf_df.to_csv(tf_csv, index=False)

print("Saved:")
print(f"  {freeform_csv}  ({len(freeform_df)} rows)")
print(f"  {tf_csv}        ({len(tf_df)} rows)")
